In [ ]:
import pandas as pd

# Read the Excel file
df = pd.read_excel('./usa_data.xlsx')

# Display the first few rows of the dataframe
print(df.head())

# Get basic information about the dataframe
print(df.info())

In [ ]:
import pandas as pd
from googletrans import Translator
import time
import concurrent.futures
import threading

local = threading.local()

# Function to get or create a thread-local translator
def get_translator():
    if not hasattr(local, 'translator'):
        local.translator = Translator()
    return local.translator

# Function to translate text with error handling and rate limiting
def translate_text(text):
    if not isinstance(text, str):
        return text
    try:
        translator = get_translator()
        time.sleep(0.1)  # Reduced sleep time
        return translator.translate(text, src='fr', dest='en').text
    except Exception as e:
        print(f"Translation error: {e}")
        return text

# Function to translate a column
def translate_column(column):
    return df[column].apply(translate_text)

# Get list of columns to translate (only text columns)
columns_to_translate = df.select_dtypes(include=['object']).columns

# Use ThreadPoolExecutor to parallelize translation
with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
    # Submit translation tasks for each column
    future_to_column = {executor.submit(translate_column, column): column for column in columns_to_translate}
    
    # Process completed tasks
    for future in concurrent.futures.as_completed(future_to_column):
        column = future_to_column[future]
        try:
            df[column] = future.result()
            print(f"Translated column: {column}")
        except Exception as exc:
            print(f'{column} generated an exception: {exc}')

# Save the translated dataframe to a new Excel file
df.to_excel('./data_translated_threaded.xlsx', index=False)

print("Translation complete. Saved as 'data_translated_threaded.xlsx'")

In [ ]:
import pandas as pd
from googletrans import Translator
import time
import concurrent.futures
import threading

local = threading.local()

# Function to get or create a thread-local translator
def get_translator():
    if not hasattr(local, 'translator'):
        local.translator = Translator()
    return local.translator

# Function to translate text with error handling and rate limiting
def translate_text(text):
    if not isinstance(text, str):
        return text
    try:
        translator = get_translator()
        time.sleep(0.1)  # Reduced sleep time
        return translator.translate(text, src='en', dest='zh-cn').text
    except Exception as e:
        print(f"Translation error: {e}")
        return text

# Function to translate a column
def translate_column(column):
    return df[column].apply(translate_text)

# Get list of columns to translate (only text columns)
columns_to_translate = df.select_dtypes(include=['object']).columns

# Use ThreadPoolExecutor to parallelize translation
with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
    # Submit translation tasks for each column
    future_to_column = {executor.submit(translate_column, column): column for column in columns_to_translate}
    
    # Process completed tasks
    for future in concurrent.futures.as_completed(future_to_column):
        column = future_to_column[future]
        try:
            df[column] = future.result()
            print(f"Translated column: {column}")
        except Exception as exc:
            print(f'{column} generated an exception: {exc}')

# Save the translated dataframe to a new Excel file
df.to_excel('./data_translated_threaded_ch.xlsx', index=False)

print("Translation complete. Saved as 'data_translated_threaded_ch.xlsx'")

In [ ]:
import pandas as pd

# Read the two Excel files
df1 = pd.read_excel('data_translated_threaded.xlsx')
df2 = pd.read_excel('data_translated_threaded_ch.xlsx')

# Determine the number of rows to process
max_rows = max(len(df1), len(df2))

# Create a new empty dataframe to store the result
result_df = pd.DataFrame()

# Alternate rows from each dataframe
for i in range(max_rows):
    if i < len(df1):
        result_df = pd.concat([result_df, df1.iloc[[i]]], ignore_index=True)
    if i < len(df2):
        result_df = pd.concat([result_df, df2.iloc[[i]]], ignore_index=True)

# Save the result to a new Excel file
result_df.to_excel('combined_alternating.xlsx', index=False)

print("Combination complete. Saved as 'combined_alternating.xlsx'")